Some Versions of libraries to download, so errors dont occur due to dependencies incompatibility

In [48]:
# !pip uninstall -y transformers peft accelerate

In [49]:
# !pip install -q \
# numpy==1.26.4 \
# pandas==2.2.2 \
# datasets==2.19.1 \
# transformers==4.55.4 \
# accelerate \
# evaluate \
# sentencepiece \
# scikit-learn \
# peft

In [50]:
import os
import time
import torch
import numpy as np
import pandas as pd

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from peft import (
    AdaLoraConfig,
    TaskType,
    get_peft_model
)

import evaluate

In [51]:
!git clone https://github.com/nyu-mll/GLUE-baselines.git
%cd GLUE-baselines
!python download_glue_data.py --data_dir glue_data --tasks CoLA

Cloning into 'GLUE-baselines'...
remote: Enumerating objects: 891, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 891 (delta 1), reused 3 (delta 1), pack-reused 886 (from 1)
Receiving objects: 100% (891/891), 1.48 MiB | 3.92 MiB/s, done.
Resolving deltas: 100% (610/610), done.
/content/GLUE-baselines/GLUE-baselines
	Completed!


In [52]:
!ls glue_data/CoLA

dev.tsv  original  test.tsv  train.tsv


Preparing the data

In [53]:
train_df = pd.read_csv(
    "glue_data/CoLA/train.tsv",
    sep="\t",
    header=None
)

train_df.head()

,0,1,2,3
0,gj04,1,NaN,"Our friends won't buy this analysis, let alone..."
1,gj04,1,NaN,One more pseudo generalization and I'm giving up.
2,gj04,1,NaN,One more pseudo generalization or I'm giving up.
3,gj04,1,NaN,"The more we study verbs, the crazier they get."
4,gj04,1,NaN,Day by day the facts are getting murkier.


In [54]:
dev_df = pd.read_csv(
    "glue_data/CoLA/dev.tsv",
    sep="\t",
    header=None
)

dev_df.head()

,0,1,2,3
0,gj04,1,NaN,The sailors rode the breeze clear of the rocks.
1,gj04,1,NaN,The weights made the rope stretch over the pul...
2,gj04,1,NaN,The mechanical doll wriggled itself loose.
3,cj99,1,NaN,"If you had eaten more, you would want less."
4,cj99,0,*,"As you eat the most, you want the least."


In [55]:
train_df = train_df[[1, 3]]
dev_df = dev_df[[1, 3]]

train_df.columns = ["label", "sentence"]
dev_df.columns = ["label", "sentence"]

In [56]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(dev_df)

In [57]:
MODEL_NAME = "microsoft/deberta-v3-base"

In [58]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [59]:
def tokenize_function(examples):

    return tokenizer(
        examples["sentence"],
        truncation=True,
        max_length=128
    )

In [60]:
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/8551 [00:00<?, ? examples/s]

In [61]:
val_dataset = val_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/1043 [00:00<?, ? examples/s]

In [62]:
train_dataset = train_dataset.rename_column(
    "label",
    "labels"
)

val_dataset = val_dataset.rename_column(
    "label",
    "labels"
)

In [63]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
AdaLoRA

In [64]:
train_batch_size = 16
epochs = 10

steps_per_epoch = len(train_dataset) // train_batch_size
total_steps = steps_per_epoch * epochs

tinit = int(0.1 * total_steps)
tfinal = int(0.8 * total_steps)

print("Total steps:", total_steps)
print("tinit:", tinit)
print("tfinal:", tfinal)

adalora_config = AdaLoraConfig(
    task_type=TaskType.SEQ_CLS,

    init_r=12,
    target_r=8,

    beta1=0.85,
    beta2=0.85,

    tinit=tinit,
    tfinal=tfinal,

    deltaT=10,

    total_step=total_steps,

    lora_alpha=16,
    lora_dropout=0.1,

    target_modules=[
        "query_proj",
        "value_proj"
    ]
)

Total steps: 5340
tinit: 534
tfinal: 4272


In [65]:
model = get_peft_model(
    model,
    adalora_config
)

In [66]:
model.print_trainable_parameters()

trainable params: 444,194 || all params: 184,867,900 || trainable%: 0.2403


In [67]:
metric = evaluate.load(
    "glue",
    "cola"
)

In [68]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(
        logits,
        axis=-1
    )

    return metric.compute(
        predictions=preds,
        references=labels
    )

In [69]:
training_args = TrainingArguments(
    output_dir="./adalora_cola",

    learning_rate=2e-4,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=10,

    weight_decay=0.01,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="matthews_correlation",

    greater_is_better=True,

    logging_steps=50,

    report_to="none"
)

In [70]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [71]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [105]:
train_dataset = train_dataset.remove_columns(["sentence"])
val_dataset = val_dataset.remove_columns(["sentence"])

print(train_dataset.column_names)

['labels', 'input_ids', 'token_type_ids', 'attention_mask']


In [106]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=data_collator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=data_collator
)

In [107]:
import torch

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=0.01
)

In [108]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

print(device)

cuda


In [109]:
import time
from tqdm.auto import tqdm

num_epochs = 10
global_step = 0

start = time.time()

model.train()

for epoch in range(num_epochs):

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for batch in progress_bar:

        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        global_step += 1

        model.base_model.update_and_allocate(global_step)

        optimizer.zero_grad()

        progress_bar.set_postfix(loss=loss.item())

end = time.time()

training_time_minutes = (end - start) / 60

print(f"Training Time: {training_time_minutes:.2f} minutes")

Epoch 1:   0%|          | 0/535 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/535 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/535 [00:00<?, ?it/s]

Epoch 4:   0%|          | 0/535 [00:00<?, ?it/s]

Epoch 5:   0%|          | 0/535 [00:00<?, ?it/s]

...

Epoch 7:   0%|          | 0/535 [00:00<?, ?it/s]

Epoch 8:   0%|          | 0/535 [00:00<?, ?it/s]

Epoch 9:   0%|          | 0/535 [00:00<?, ?it/s]

Epoch 10:   0%|          | 0/535 [00:00<?, ?it/s]

Training Time: 10.08 minutes


Trying to find where the dynamically allocated ranks are stored

In [111]:
print(model.peft_config["default"].rank_pattern)

{'deberta.encoder.layer.0.attention.self.query_proj.lora_E.default': [False, False, False, False, False, False, False, False, False, False, False, False], 'deberta.encoder.layer.0.attention.self.value_proj.lora_E.default': [True, True, True, True, False, True, True, True, True, True, True, True], 'deberta.encoder.layer.1.attention.self.query_proj.lora_E.default': [False, False, True, False, True, False, False, False, False, False, False, False], 'deberta.encoder.layer.1.attention.self.value_proj.lora_E.default': [True, True, True, True, True, True, True, False, True, True, False, True], 'deberta.encoder.layer.2.attention.self.query_proj.lora_E.default': [False, False, True, False, False, False, False, False, False, True, False, True], 'deberta.encoder.layer.2.attention.self.value_proj.lora_E.default': [True, False, True, True, True, True, False, False, True, True, True, True], 'deberta.encoder.layer.3.attention.self.query_proj.lora_E.default': [True, True, False, True, False, False, Tr

In [112]:
total_rank = 0
num_modules = 0

for key, mask in model.peft_config["default"].rank_pattern.items():
    rank = sum(mask)

    print(key, rank)

    total_rank += rank
    num_modules += 1

print("\nAverage Effective Rank:", total_rank / num_modules)
print("Total Effective Rank:", total_rank)

deberta.encoder.layer.0.attention.self.query_proj.lora_E.default 0
deberta.encoder.layer.0.attention.self.value_proj.lora_E.default 11
deberta.encoder.layer.1.attention.self.query_proj.lora_E.default 2
deberta.encoder.layer.1.attention.self.value_proj.lora_E.default 10
deberta.encoder.layer.2.attention.self.query_proj.lora_E.default 3
deberta.encoder.layer.2.attention.self.value_proj.lora_E.default 9
deberta.encoder.layer.3.attention.self.query_proj.lora_E.default 5
deberta.encoder.layer.3.attention.self.value_proj.lora_E.default 10
deberta.encoder.layer.4.attention.self.query_proj.lora_E.default 2
deberta.encoder.layer.4.attention.self.value_proj.lora_E.default 11
deberta.encoder.layer.5.attention.self.query_proj.lora_E.default 5
deberta.encoder.layer.5.attention.self.value_proj.lora_E.default 12
deberta.encoder.layer.6.attention.self.query_proj.lora_E.default 3
deberta.encoder.layer.6.attention.self.value_proj.lora_E.default 12
deberta.encoder.layer.7.attention.self.query_proj.lora_E

In [113]:
import torch
from tqdm.auto import tqdm

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for batch in tqdm(val_loader):

        labels = batch["labels"]

        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        outputs = model(**batch)

        preds = torch.argmax(outputs.logits, dim=-1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

  0%|          | 0/66 [00:00<?, ?it/s]

In [114]:
from sklearn.metrics import matthews_corrcoef

mcc = matthews_corrcoef(
    all_labels,
    all_preds
)

print("MCC:", mcc)

MCC: 0.6586120970818117


In [115]:
total_rank = 0
num_modules = 0

for key, mask in model.peft_config["default"].rank_pattern.items():

    rank = sum(mask)

    total_rank += rank
    num_modules += 1

avg_rank = total_rank / num_modules

print("Average Effective Rank:", avg_rank)
print("Total Effective Rank:", total_rank)

Average Effective Rank: 8.0
Total Effective Rank: 192


In [116]:
trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print("Trainable Params:", trainable_params)

print(
    "Trainable Percentage:",
    100 * trainable_params / total_params
)

Trainable Params: 444194
Trainable Percentage: 0.24027643522753275


In [117]:
adalora_results = {
    "Method": "AdaLoRA",
    "MCC": float(mcc),
    "Trainable_Params": trainable_params,
    "Average_Effective_Rank": float(avg_rank),
    "Total_Effective_Rank": int(total_rank),
    "Training_Time_Min": float(training_time_minutes)
}

adalora_results

{'Method': 'AdaLoRA',
 'MCC': 0.6586120970818117,
 'Trainable_Params': 444194,
 'Average_Effective_Rank': 8.0,
 'Total_Effective_Rank': 192,
 'Training_Time_Min': 10.081978917121887}

In [118]:
print("=" * 60)
print("AdaLoRA Final Results")
print(f"MCC: {mcc:.4f}")
print(f"Training Time: {training_time_minutes:.2f} minutes")
print(f"Trainable Params: {trainable_params:,}")
print(f"Average Effective Rank: {avg_rank:.2f}")
print(f"Total Effective Rank: {total_rank}")
print("=" * 60)

AdaLoRA Final Results
MCC: 0.6586
Training Time: 10.08 minutes
Trainable Params: 444,194
Average Effective Rank: 8.00
Total Effective Rank: 192


In [119]:
from google.colab import drive
drive.mount('/content/drive')

model.save_pretrained("/content/drive/MyDrive/adalora_cola_final")
tokenizer.save_pretrained("/content/drive/MyDrive/adalora_cola_final")

Mounted at /content/drive


('/content/drive/MyDrive/adalora_cola_final/tokenizer_config.json',
 '/content/drive/MyDrive/adalora_cola_final/special_tokens_map.json',
 '/content/drive/MyDrive/adalora_cola_final/spm.model',
 '/content/drive/MyDrive/adalora_cola_final/added_tokens.json',
 '/content/drive/MyDrive/adalora_cola_final/tokenizer.json')